# Эксперименты: ResNet для прогнозирования финансовой динамики

**Автор:** Миндрин Тимофей Дмитриевич

В данном ноутбуке проводится сравнительный анализ остаточной нейронной сети (ResNet) с baseline-моделями (LSTM, RandomForest) на задаче классификации направления движения цен российских акций.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

from src.config import DATA_CFG, MODEL_CFG, PATHS
from src.data import prepare_dataset
from src.features import build_features, create_sequences, prepare_ml_dataset
from src.models import ResNetTimeSeries, LSTMClassifier
from src.train import create_dataloaders, train_model
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_curve, save_metrics
from src.utils import set_seed, setup_logging

set_seed(42)
setup_logging()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {device}')

## 1. Загрузка данных

In [ ]:
datasets = prepare_dataset()
ticker = 'SBER.ME'
df = datasets[ticker]
df.head()

## 2. Feature Engineering

In [ ]:
features = build_features(df)
features.head()

## 3. Подготовка выборок для глубокого обучения (последовательности)

In [ ]:
# Целевая переменная: цена завтра выше сегодня
target = (features['close'].shift(-1) > features['close']).astype(int)
features_clean = features.iloc[:-1]
target_clean = target.iloc[:-1]

X_seq, y_seq = create_sequences(features_clean, target_clean, window_size=DATA_CFG.window_size)
print(f'X_seq shape: {X_seq.shape}, y_seq shape: {y_seq.shape}')

# Разделение с учётом временной структуры (без shuffle!)
n = len(X_seq)
train_end = int(n * (1 - DATA_CFG.test_size - DATA_CFG.val_size))
val_end = int(n * (1 - DATA_CFG.test_size))

X_train, y_train = X_seq[:train_end], y_seq[:train_end]
X_val, y_val = X_seq[train_end:val_end], y_seq[train_end:val_end]
X_test, y_test = X_seq[val_end:], y_seq[val_end:]

# Нормализация по train
scaler = StandardScaler()
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
scaler.fit(X_train_2d)

def scale_3d(x):
    shape = x.shape
    return scaler.transform(x.reshape(-1, shape[-1])).reshape(shape)

X_train = scale_3d(X_train)
X_val = scale_3d(X_val)
X_test = scale_3d(X_test)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## 4. Обучение ResNet

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test)

resnet = ResNetTimeSeries(
    input_channels=X_train.shape[-1],
    num_classes=2,
    block_channels=MODEL_CFG.resnet_filters,
    num_blocks=MODEL_CFG.resnet_blocks,
    kernel_size=MODEL_CFG.resnet_kernel_size,
    dropout=MODEL_CFG.dropout
)

history_resnet = train_model(
    resnet, train_loader, val_loader,
    device=device,
    save_dir=PATHS.results
)

## 5. Обучение LSTM (baseline)

In [ ]:
lstm = LSTMClassifier(
    input_size=X_train.shape[-1],
    hidden_size=MODEL_CFG.lstm_hidden,
    num_layers=MODEL_CFG.lstm_layers,
    num_classes=2,
    dropout=MODEL_CFG.dropout
)

history_lstm = train_model(
    lstm, train_loader, val_loader,
    device=device,
    save_dir=PATHS.results
)

## 6. RandomForest (baseline без окон)

In [ ]:
X_ml, y_ml = prepare_ml_dataset(features)
X_ml_train, X_ml_test, y_ml_train, y_ml_test = train_test_split(
    X_ml, y_ml, test_size=DATA_CFG.test_size + DATA_CFG.val_size, shuffle=False
)

scaler_rf = StandardScaler()
X_ml_train_scaled = scaler_rf.fit_transform(X_ml_train)
X_ml_test_scaled = scaler_rf.transform(X_ml_test)

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_ml_train_scaled, y_ml_train)
rf_pred = rf.predict(X_ml_test_scaled)
rf_prob = rf.predict_proba(X_ml_test_scaled)[:, 1]

rf_metrics = {
    'accuracy': accuracy_score(y_ml_test, rf_pred),
    'f1': f1_score(y_ml_test, rf_pred),
    'roc_auc': roc_auc_score(y_ml_test, rf_prob)
}
print('RandomForest metrics:', rf_metrics)

## 7. Оценка и сравнение моделей

In [ ]:
metrics_resnet, y_true, y_pred_resnet, y_prob_resnet = evaluate_model(resnet, test_loader, device=device)
metrics_lstm, _, y_pred_lstm, y_prob_lstm = evaluate_model(lstm, test_loader, device=device)

# Сводная таблица
comparison = pd.DataFrame({
    'ResNet': metrics_resnet,
    'LSTM': metrics_lstm,
    'RandomForest': rf_metrics
})
comparison = comparison.round(4)
display(comparison)
comparison.to_csv(PATHS.results / 'comparison.csv')

## 8. Визуализация

In [ ]:
# Кривые обучения
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(history_resnet['train_loss'], label='ResNet train')
ax[0].plot(history_resnet['val_loss'], label='ResNet val')
ax[0].set_title('ResNet: кривые обучения')
ax[0].set_xlabel('Эпоха')
ax[0].set_ylabel('Loss')
ax[0].legend()
ax[0].grid(True)

ax[1].plot(history_lstm['train_loss'], label='LSTM train')
ax[1].plot(history_lstm['val_loss'], label='LSTM val')
ax[1].set_title('LSTM: кривые обучения')
ax[1].set_xlabel('Эпоха')
ax[1].set_ylabel('Loss')
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.savefig(PATHS.results / 'learning_curves.png', dpi=300)
plt.show()

In [ ]:
# Матрицы ошибок
plot_confusion_matrix(y_true, y_pred_resnet, save_path=PATHS.results / 'cm_resnet.png')
plot_confusion_matrix(y_true, y_pred_lstm, save_path=PATHS.results / 'cm_lstm.png')

In [ ]:
# ROC-кривые
plot_roc_curve(y_true, y_prob_resnet, save_path=PATHS.results / 'roc_resnet.png')
plot_roc_curve(y_true, y_prob_lstm, save_path=PATHS.results / 'roc_lstm.png')

## 9. Сохранение результатов

In [ ]:
save_metrics(metrics_resnet, PATHS.results / 'metrics_resnet.json')
save_metrics(metrics_lstm, PATHS.results / 'metrics_lstm.json')
save_metrics(rf_metrics, PATHS.results / 'metrics_rf.json')

print('Все результаты сохранены в', PATHS.results)